In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
#Dataset and Dataloaders
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms

# imgae => scale(0,1) => normalize => (-1,1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform = transform)
testset = CIFAR10(root="./data", train= False, download=True, transform=transform)

Files already downloaded and verified
Files already downloaded and verified


In [3]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True) 
testloader = DataLoader(testset, batch_size=64)

### Build the CNN

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32 ,kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernel_size = 2, Sride_value=2

            nn.Conv2d(32,64 ,kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernel_size = 2, Sride_value=2

            nn.Conv2d(64,128 ,kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernel_size = 2, Sride_value=2
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
            x=self.conv_layers(x)
            x=x.view(x.size(0), -1) #flattening
            x=self.fc_layers(x)
            
            return x

In [5]:
model = CNN()

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN

In [7]:
epochs =10

for epoch in range(epochs):
    epoch_training_loss =0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        outputs = model.forward(images) #FP
        loss = criterion(outputs, labels) # loss function
        loss.backward() #BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch = {epoch+1}/{epochs} & loss ={epoch_training_loss/ len(trainloader)}")

epoch = 1/10 & loss =1.3472132746063534
epoch = 2/10 & loss =0.9302542545758855
epoch = 3/10 & loss =0.7431680771243542
epoch = 4/10 & loss =0.614145087235419
epoch = 5/10 & loss =0.511055736552419
epoch = 6/10 & loss =0.41021367629318284
epoch = 7/10 & loss =0.3252986978710917
epoch = 8/10 & loss =0.2515231726781639
epoch = 9/10 & loss =0.19812566352069683
epoch = 10/10 & loss =0.150131542728666


### Evaluating the CNN


In [11]:
correct_labels = 0
total_labels = 0

model.eval()
with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)
print(f"accuracy = {correct_labels / total_labels*100}")

accuracy = 74.32
